# 11 — Cohort B — apply the federated result, and compare

| | |
|---|---|
| **runs at** | cohort B |
| **reads** | `cohort-B.csv`, `federated_pvclust_edges.csv`, `cohort-B_edges.csv` |
| **writes** | `cohort-B_from-federated_*.csv`, figures |

**There is no single global answer.** Each cohort gets its own outcome, and the
federated tree is the channel that carries the pooled information back here.

The question this answers is not "what are the federated clusters" — step 09
already said. It is **"what do they look like in my data, and was joining worth
it?"**

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
# The SLE data is not in the repository. Download it from Zenodo
# (doi:10.5281/zenodo.20342569), unpack it, and point SLE at the folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 80        # 80 objects keeps p < n in BOTH cohorts
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
else:
    # A stand-in of the same shape, so every cell runs without the download:
    # two batches of unequal size, a case/control split, and a few proteins
    # measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    FEATURE_MAP = 'feature_metadata.txt'
print(f"{'SLE data' if REAL else 'stand-in'}: n_boot={NBOOT}, top_variable={TOPVAR}")

In [ ]:
run(['pvclust-py', 'apply-edges', '--project', 'cohort-B',
     '--matrix', 'cohort-B.csv', '--log2',
     '--metadata', 'cohort-B_meta.csv',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--shared-features', 'shared_features.csv',
     '--federated-edges', 'federated_pvclust_edges.csv',
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol',
     '--dist', 'minkowski', '--linkage', 'ward.D2',
     '--n-boot', str(NBOOT), '--alpha', '0.95', '--plot'])

## Where federating helped most

`au_local` is this cohort's own support for a cluster; `au_federated` is the
pooled support. A cluster with **high federated AU and low local AU** is one
this cohort could not have found alone — that is the gain, made concrete and
cluster by cluster rather than as a single headline number.

In [ ]:
sup = pd.read_csv('cohort-B_from-federated_edge_support.csv')
print(sup.head(10)[['n_members', 'au_local', 'au_federated', 'gain']]
      .to_string(index=False))

## Standalone versus federated, side by side

The honest comparison. `in_solo_tree` says whether this cohort's own tree from
step 04/05 contained the cluster at all.

**A negative median gain is a real result, not a failure.** It would mean the
other cohort's data pulled the pooled tree away from what this cohort sees,
which is worth knowing before anyone builds on the pooled answer.

In [ ]:
agree = pd.read_csv('cohort-B_from-federated_agreement.csv')
print(agree.to_string(index=False))

In [ ]:
show('cohort-B_from-federated_heatmap.png')

## The module scores

Each well-supported federated cluster reduced to one score per sample, so this
cohort can carry a federation-validated feature space into whatever it does
next.

Modules are **non-overlapping** and come from the *smallest* well-supported
clusters, not the largest. The catalogue is a tree, so its clusters are nested;
averaging every cluster that passes the threshold would give near-duplicate
columns, and taking the largest would collapse everything to one near-root
module. Anything in no module is kept as itself, so the space is a reduction of
the original rather than a copy of it.

In [ ]:
mods = pd.read_csv('cohort-B_from-federated_modules.csv')
grouped = mods[mods.n_members > 1]
print(f'{len(grouped)} modules over {grouped.n_members.sum()} proteins; '
      f'{len(mods) - len(grouped)} proteins kept as themselves')
print()
for _, r in grouped.head(8).iterrows():
    print(f'  n={r.n_members}  AU={r.au:.3f}  {r.members[:64]}')

---
**The cycle ends here.** There is no iteration loop, and that is deliberate:
the pooled distance is computed exactly from summed statistics, so pooling the
same statistics twice returns an identical tree. Nothing moves, so nothing
converges. `oadr-cpep` can iterate because each round refits local models from
the federated coefficients, and the inputs genuinely change; there is no
analogue here.

Clustering also has no error to plot — no outcome, no residual. What can be
reported is the local-versus-federated AU per cluster, which is the table above.